# Lung Cancer Detection and Diagnostic Support System
## Member 2: Machine Learning Classification & Multimodal Clinical Risk Engine

---
### Project Context & Role Delegation
- **Member 1 (Upstream)**: Image processing & feature extraction from chest CT scans (GLCM texture contrast, segmented nodule area).
- **Member 2 (Current)**: Model training, evaluation, comparison across Logistic Regression, Random Forest, and SVM; export production `.pkl` and `features.json`; implementation of composite multimodal risk score combining CT scan model confidence (60%) with clinical symptom scores (40%).
- **Member 3 (Downstream)**: API integration of the serialized pipeline into a production FastAPI endpoint for clinical decision support.


### 1. Environment Setup & Dependency Imports

In [ ]:
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Visualization configuration
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'font.size': 11})
print('[OK] Core libraries imported successfully.')

### 2. Data Loading & Exploratory Data Analysis (EDA)
We inspect the extracted feature dataset produced by Member 1 (`features.csv`).

In [ ]:
# Locate features.csv across candidate paths
csv_candidates = [
    os.path.join(os.path.expanduser('~'), 'OneDrive', 'Desktop', 'features.csv'),
    os.path.join(os.path.expanduser('~'), 'Desktop', 'features.csv'),
    'features.csv'
]
csv_path = next((p for p in csv_candidates if os.path.exists(p)), 'features.csv')
print(f'Loading data from: {csv_path}')

df = pd.read_csv(csv_path)
print(f'Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns\n')
print('Columns and Types:')
print(df.dtypes)
print('\nMissing Values Count:')
print(df.isnull().sum())
print('\nTarget Class Balance (label):')
print(df['label'].value_counts())
print('\nClass Proportions (%):')
print(df['label'].value_counts(normalize=True) * 100)
df.head()

### 3. Feature Selection & Stratified Train-Test Split (80/20)
- Filter non-feature metadata (`image_id`, `split`).
- Target mapped to binary: `1 = Cancer`, `0 = Normal`.
- Stratified split preserves the 78.5% : 21.5% class balance.

In [ ]:
feature_cols = ['contrast', 'area']
X = df[feature_cols].copy()
y = df['label'].str.strip().str.lower().map({'cancer': 1, 'normal': 0}).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training Set: {len(X_train)} samples ({y_train.sum()} Cancer, {len(y_train)-y_train.sum()} Normal)')
print(f'Testing Set : {len(X_test)} samples ({y_test.sum()} Cancer, {len(y_test)-y_test.sum()} Normal)')

### 4. Model Training & Benchmarking
Training three standard classifiers using Scikit-Learn Pipelines:
1. **Logistic Regression** (with `StandardScaler`)
2. **Random Forest** (100 trees, balanced weights)
3. **Support Vector Machine (SVM)** (RBF kernel, `StandardScaler`, Platt scaling enabled)

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', RandomForestClassifier(n_estimators=100, max_depth=6, class_weight='balanced', random_state=42))
    ]),
    'Support Vector Machine': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', SVC(kernel='rbf', C=1.0, probability=True, class_weight='balanced', random_state=42))
    ])
}

results = {}
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    results[name] = {
        'pipeline': pipeline,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1_score': f1_score(y_test, y_pred, zero_division=0),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'y_pred': y_pred,
        'y_prob': y_prob
    }

summary = pd.DataFrame([
    {
        'Model': m,
        'Accuracy': f"{results[m]['accuracy']*100:.2f}%",
        'Precision': f"{results[m]['precision']*100:.2f}%",
        'Recall': f"{results[m]['recall']*100:.2f}%",
        'F1-Score': f"{results[m]['f1_score']*100:.2f}%"
    } for m in results
])
best_model_name = max(results, key=lambda k: (results[k]['f1_score'], results[k]['accuracy']))
print(f'Winning Model: {best_model_name}')
summary

### 5. Diagnostic Evaluation & Plots
- Confusion Matrix Heatmaps for all 3 models
- Model Accuracy Comparison
- Classification Report for winning model

In [ ]:
target_labels = ['Normal', 'Cancer']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, res) in zip(axes, results.items()):
    sns.heatmap(res['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
                xticklabels=target_labels, yticklabels=target_labels, cbar=False, ax=ax,
                annot_kws={'size': 13, 'weight': 'bold'})
    ax.set_title(f"{name}\nAcc: {res['accuracy']:.2%} | F1: {res['f1_score']:.4f}", fontsize=12)
    ax.set_xlabel('Predicted Diagnosis')
    ax.set_ylabel('True Diagnosis')
plt.suptitle('Test Set Confusion Matrices Across Candidate Models', fontsize=14, weight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Accuracy Comparison Bar Chart
plt.figure(figsize=(7, 4.5))
accs = [results[m]['accuracy']*100 for m in models]
bars = plt.bar(list(models.keys()), accs, color=['#2b5c8f', '#388e3c', '#d9534f'], width=0.5, edgecolor='black')
plt.title('Test Set Accuracy Comparison', weight='bold')
plt.ylabel('Accuracy (%)')
plt.ylim(0, 105)
for bar in bars:
    plt.annotate(f'{bar.get_height():.2f}%', (bar.get_x() + bar.get_width()/2, bar.get_height()),
                 xytext=(0, 4), textcoords='offset points', ha='center', weight='bold')
plt.tight_layout()
plt.show()

print(f'Classification Report for Best Model ({best_model_name}):')
print(classification_report(y_test, results[best_model_name]['y_pred'], target_names=['Normal (0)', 'Cancer (1)'], digits=4))

### 6. Artifact Serialization for Member 3 (FastAPI Backend)
We export `best_lung_cancer_model.pkl` and `features.json` so Member 3 can deploy inference with zero friction.

In [ ]:
best_pipeline = results[best_model_name]['pipeline']
joblib.dump(best_pipeline, 'best_lung_cancer_model.pkl')

schema = {
    'project': 'Lung Cancer Detection and Diagnostic Support System',
    'author': 'Member 2 - ML Pipeline',
    'best_model': best_model_name,
    'feature_order': feature_cols,
    'n_features': len(feature_cols),
    'target_mapping': {'0': 'Normal', '1': 'Cancer'},
    'risk_formula': 'risk_score = (model_confidence * 0.6) + (symptom_score * 0.4)',
    'risk_thresholds': {'low': '< 40.0', 'medium': '40.0 - 70.0', 'high': '> 70.0'}
}
with open('features.json', 'w') as f:
    json.dump(schema, f, indent=4)
print('[OK] Serialized best_lung_cancer_model.pkl and features.json successfully.')

### 7. Multimodal Clinical Risk Scoring Engine
$$\text{risk\_score} = (\text{model\_confidence} \times 0.6) + (\text{symptom\_score} \times 0.4)$$

**Weighting Rationale for Clinical Symptoms (100 Points Total):**
1. `smoking_history` (25 pts): Primary cause of lung cancer (~85% of cases).
2. `age` (15 pts): Age >= 60 (15 pts), 50-59 (10 pts), <50 (2-5 pts).
3. `cough` (15 pts): Hallmark symptom of bronchogenic carcinoma.
4. `breathlessness` (15 pts): Dyspnea indicates bronchial obstruction or effusion.
5. `chest_pain` (10 pts): Thoracic wall invasion.
6. `weight_loss` (10 pts): Malignant cachexia.
7. `fatigue` (10 pts): Systemic cytokine tumor response.

In [ ]:
def calculate_symptom_score(patient_data: dict) -> float:
    def is_present(val):
        if val is None: return False
        if isinstance(val, (bool, np.bool_)): return bool(val)
        if isinstance(val, (int, float)): return val > 0
        return str(val).strip().lower() in ['yes', 'y', 'true', '1', 'positive', 'present', 'severe']
    score = 0.0
    smk = patient_data.get('smoking_history', 'No')
    if str(smk).lower() in ['current', 'heavy', 'yes', 'true', '1']:
        score += 25.0
    elif str(smk).lower() in ['former', 'past', 'light']:
        score += 15.0
    age = float(patient_data.get('age', 45))
    score += 15.0 if age >= 60 else (10.0 if age >= 50 else (5.0 if age >= 40 else 2.0))
    if is_present(patient_data.get('cough')): score += 15.0
    if is_present(patient_data.get('breathlessness')): score += 15.0
    if is_present(patient_data.get('chest_pain')): score += 10.0
    if is_present(patient_data.get('weight_loss')): score += 10.0
    if is_present(patient_data.get('fatigue')): score += 10.0
    return round(float(np.clip(score, 0.0, 100.0)), 2)

def calculate_risk_score(model_confidence: float, patient_data: dict) -> float:
    conf_100 = model_confidence * 100.0 if model_confidence <= 1.0 else model_confidence
    symptom_score = calculate_symptom_score(patient_data)
    return round(float(np.clip((conf_100 * 0.6) + (symptom_score * 0.4), 0.0, 100.0)), 2)

def get_full_risk_assessment(model, scaler, patient_data, feature_cols=['contrast', 'area']):
    x = np.array([float(patient_data[c]) for c in feature_cols]).reshape(1, -1)
    prob = float(model.predict_proba(x)[0][1])
    symptom_score = calculate_symptom_score(patient_data)
    risk_score = calculate_risk_score(prob, patient_data)
    if risk_score < 40.0:
        level, rec = 'Low', 'Low clinical risk. Routine lifestyle counseling & standard annual follow-up.'
    elif risk_score <= 70.0:
        level, rec = 'Medium', 'Moderate risk. Pulmonologist consultation & follow-up low-dose CT in 3-6 months.'
    else:
        level, rec = 'High', 'HIGH CLINICAL RISK! Urgent multidisciplinary oncology referral for biopsy & staging.'
    return {
        'model_confidence': round(prob * 100.0, 2),
        'symptom_score': symptom_score,
        'risk_score': risk_score,
        'risk_level': level,
        'recommendation': rec
    }
print('[OK] Risk assessment functions successfully defined.')

### 8. End-to-End Clinical Demonstration on Test Patients

In [ ]:
demo_cases = [
    {
        'id': 'Case 1 (Test #377 - Low Risk)',
        'contrast': float(X_test.loc[377]['contrast']),
        'area': float(X_test.loc[377]['area']),
        'age': 34, 'smoking_history': 'No', 'cough': 'No',
        'breathlessness': 'No', 'chest_pain': 'No', 'weight_loss': 'No', 'fatigue': 'No'
    },
    {
        'id': 'Case 2 (Test #488 - Medium Risk)',
        'contrast': float(X_test.loc[488]['contrast']),
        'area': float(X_test.loc[488]['area']),
        'age': 56, 'smoking_history': 'Former', 'cough': 'Yes',
        'breathlessness': 'No', 'chest_pain': 'No', 'weight_loss': 'No', 'fatigue': 'Yes'
    },
    {
        'id': 'Case 3 (Test #925 - High Risk)',
        'contrast': float(X_test.loc[925]['contrast']),
        'area': float(X_test.loc[925]['area']),
        'age': 67, 'smoking_history': 'Current', 'cough': 'Yes',
        'breathlessness': 'Yes', 'chest_pain': 'Yes', 'weight_loss': 'Yes', 'fatigue': 'Yes'
    }
]

for c in demo_cases:
    res = get_full_risk_assessment(best_pipeline, None, c)
    print(f"=== {c['id']} ===")
    print(f"CT Features      : Contrast = {c['contrast']:.4f}, Area = {c['area']:.1f} px")
    print(f"Model Confidence : {res['model_confidence']}%")
    print(f"Symptom Score    : {res['symptom_score']} / 100")
    print(f"Composite Risk   : {res['risk_score']} / 100  -> [ {res['risk_level'].upper()} RISK ]")
    print(f"Action Plan      : {res['recommendation']}\n")